## 1. Mount Google Drive
Ensure outputs are seamlessly backed up.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/afriqa_outputs


## 2. Install Required Dependencies

In [ ]:
!git clone https://github.com/OmondiKevin/afriqa-entity-aware-qa.git
%cd afriqa-entity-aware-qa
!git pull
!pip install -e .
!pip install sentence-transformers sentencepiece protobuf peft

## 3. Configure Environment
Symlink our output directory to Google Drive.

In [ ]:
import os
import torch
from transformers import set_seed

set_seed(42)

if os.path.exists("outputs"):
    !rm -rf outputs
!ln -s /content/drive/MyDrive/afriqa_outputs outputs


## 4. Load Dataset
Download original AfriQA and MasakhaNER JSONs.

In [ ]:
!python scripts/00_download_and_subset.py --config configs/default.yaml

## 5. Preprocess Datasets
Format raw JSONs to Seq2Seq tasks and upsample QA to fix class imbalance (Phase 1 NER vs Phase 2 QA).

In [ ]:
!python scripts/01_prepare_qa_data.py --config configs/default.yaml
!python scripts/01b_prepare_multitask_data.py --config configs/default.yaml

## 6. Configure Training (`mT5-base`)
Verify that `default.yaml` specifies `google/mt5-base` as the model (which it does by default). We quickly print it out here to confirm.

In [ ]:
!cat configs/default.yaml | grep -A 3 "model:"

## 7. Train mT5-Base Model
Execute the Multitask Sequential Pipeline directly. (Train NER -> Fine-tune QA)

In [ ]:
!python scripts/03_train_multitask_qa.py --config configs/default.yaml --sequential

## 8. Evaluate mT5-Base Model
Generates predictions isolating the QA samples securely using `--qa_only`.

In [ ]:
!python scripts/04_eval_predictions.py --config configs/default.yaml --pred_path outputs/predictions/multitask_mt5_test.jsonl --qa_only

## 9. Save Best Checkpoint
Because we established the Google Drive symlink in Cell 3, the `Seq2SeqTrainer` inside `scripts/03` already automatically saved the best performing checkpoint (eval_loss) and 1 past checkpoint to Drive during training. This cell verifies it is successfully synced.

In [ ]:
!ls -lh /content/drive/MyDrive/afriqa_outputs/checkpoints/multitask_mt5

## 10. Display Final Metrics
Output the Exact Match (EM), Token-F1, and Semantic Similarity evaluation metrics generated from Cell 8.

In [ ]:
import json

metrics_path = "outputs/metrics/multitask_mt5_test_qa_only_metrics.json"
if os.path.exists(metrics_path):
    with open(metrics_path, "r") as f:
        m = json.load(f)
        print(json.dumps(m["overall"], indent=4))
else:
    print("Metrics file not found. Ensure evaluation succeeded!")
